In [1]:
import sys

sys.path.append("..")  # Project root, so `src` is importable from notebooks/.


import torch

from src.ops import masked_softmax

## Transpose Q / K / V for parellel computation

So we have our Key, Value Pairs and Queries in Dot Product Attention. In DPA we calculate the scores first : 

Dimensions : 

```
queries : (batch_size, num_queries, len_query)
keys : (batch_size, num_key_val_pairs, len_key)
values : (batch_size, num_key_val_pairs, len_value)
```

In [2]:
batch_size = 2
num_queries = 2
num_hiddens = 4
num_heads = 2

# For each head : features = num_hiddens / num_heads
features_per_head = num_hiddens / num_heads

In [3]:
X = torch.rand((batch_size, num_queries, num_hiddens))
print(X.shape)
X

torch.Size([2, 2, 4])


tensor([[[0.8689, 0.7728, 0.3215, 0.2870],
         [0.7767, 0.6600, 0.7224, 0.2752]],

        [[0.8881, 0.5932, 0.2905, 0.3963],
         [0.2640, 0.7129, 0.5013, 0.1284]]])

In [4]:
# The -1 tells PyTorch to work out that size: 4 / 2 = 2. 
# Only the last axis changes. Each 4-wide row is cut into 2 rows of 2:
X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
X

tensor([[[[0.8689, 0.7728],
          [0.3215, 0.2870]],

         [[0.7767, 0.6600],
          [0.7224, 0.2752]]],


        [[[0.8881, 0.5932],
          [0.2905, 0.3963]],

         [[0.2640, 0.7129],
          [0.5013, 0.1284]]]])

In [5]:
assert X.shape[0] == batch_size
assert X.shape[1] == num_queries
assert X.shape[2] == num_hiddens / num_heads
assert X.shape[3] == features_per_head

X.shape

torch.Size([2, 2, 2, 2])

Current dimension of X : 

```
(batch_size, num_queries, num_heads, features_per_head)
```

Now we swap the num_queries for num_heads so that we can do batch matrix multiplication.

In [6]:
X = X.permute(0, 2, 1, 3)
X

tensor([[[[0.8689, 0.7728],
          [0.7767, 0.6600]],

         [[0.3215, 0.2870],
          [0.7224, 0.2752]]],


        [[[0.8881, 0.5932],
          [0.2640, 0.7129]],

         [[0.2905, 0.3963],
          [0.5013, 0.1284]]]])

Now we can merge the batch_size and num_heads as we need to parellelize as much as possible 

New dimension : 

```
(batch_size * num_heads, num_queries, features_per_head)
```

In [7]:
X = X.reshape(-1, X.shape[2], X.shape[3])
X

tensor([[[0.8689, 0.7728],
         [0.7767, 0.6600]],

        [[0.3215, 0.2870],
         [0.7224, 0.2752]],

        [[0.8881, 0.5932],
         [0.2640, 0.7129]],

        [[0.2905, 0.3963],
         [0.5013, 0.1284]]])

For attention now we calculate the scores by doing : 

$$
softmax(\frac{QK}{\sqrt(d)})V
$$

- We need to change the shape of Keys : batch_size, num_key_val_pairs, len_key) -> (batch_size, len_keys, num_key_val_pairs)
- Now we multiply 

In [23]:
def transpose_qkv(X: torch.Tensor, num_heads: int) -> torch.Tensor:
    """Transposition for parallel computation of multiple attention heads."""
    # input X : (batch_size, num_queries/ num_key_val_pairs, num_hiddens)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)
    X = X.permute(0, 2, 1, 3)
    return X.reshape(-1, X.shape[2], X.shape[3])

Q = transpose_qkv(torch.rand((batch_size, num_queries, num_hiddens)), num_heads)
K = transpose_qkv(torch.rand((batch_size, num_queries, num_hiddens)), num_heads)
V = transpose_qkv(torch.rand((batch_size, num_queries, num_hiddens)), num_heads)

print(Q.shape)

torch.Size([4, 2, 2])


In [24]:
scores = torch.bmm(Q, K.transpose(1, 2)) / Q.shape[-1]
print(scores.shape)
scores

torch.Size([4, 2, 2])


tensor([[[0.2648, 0.3261],
         [0.2516, 0.4559]],

        [[0.2034, 0.1603],
         [0.4501, 0.4524]],

        [[0.2664, 0.3433],
         [0.5273, 0.6606]],

        [[0.3141, 0.0825],
         [0.2918, 0.0973]]])

In [25]:
attention_weights = masked_softmax(scores, None)

In [26]:
attention_weights.shape

torch.Size([4, 2, 2])

In [27]:
attention = torch.bmm(attention_weights, V)

In [29]:
print(attention)
# (batch_size * num_heads, num_queries, features_per_head)
attention.shape

tensor([[[0.2078, 0.1400],
         [0.2167, 0.1450]],

        [[0.5057, 0.1862],
         [0.4960, 0.1849]],

        [[0.8614, 0.1796],
         [0.8654, 0.1748]],

        [[0.8018, 0.8053],
         [0.8042, 0.8077]]])


torch.Size([4, 2, 2])

Now we calculated the attention but in practice for Self Attention we have weights which need to be multiplied basically a Linear Layer. 
So what we need to do after calculation of attention is reverse the shape changes in transpose_qkv and multiply with output weights.

In [32]:
Y = torch.rand(4, 2, 2)
Y

tensor([[[0.0724, 0.0133],
         [0.5216, 0.7566]],

        [[0.1172, 0.0107],
         [0.7919, 0.7580]],

        [[0.2244, 0.3559],
         [0.5760, 0.9498]],

        [[0.1913, 0.3619],
         [0.7983, 0.6765]]])

In [36]:
Y = Y.reshape(-1, num_heads, Y.shape[1], Y.shape[2])

print(Y.shape)
# (batch_size, num_heads, num_queries, features_per_head)
Y

torch.Size([2, 2, 2, 2])


tensor([[[[0.0724, 0.0133],
          [0.5216, 0.7566]],

         [[0.1172, 0.0107],
          [0.7919, 0.7580]]],


        [[[0.2244, 0.3559],
          [0.5760, 0.9498]],

         [[0.1913, 0.3619],
          [0.7983, 0.6765]]]])

In [37]:
Y = Y.permute(0, 2, 1, 3)
# (batch_size, num_queries, num_heads, features_per_head)
Y

tensor([[[[0.0724, 0.0133],
          [0.1172, 0.0107]],

         [[0.5216, 0.7566],
          [0.7919, 0.7580]]],


        [[[0.2244, 0.3559],
          [0.1913, 0.3619]],

         [[0.5760, 0.9498],
          [0.7983, 0.6765]]]])

In [39]:
Y = Y.reshape(Y.shape[0], Y.shape[1], -1)
# (batch_size, num_queries, num_hidden)
Y

tensor([[[0.0724, 0.0133, 0.1172, 0.0107],
         [0.5216, 0.7566, 0.7919, 0.7580]],

        [[0.2244, 0.3559, 0.1913, 0.3619],
         [0.5760, 0.9498, 0.7983, 0.6765]]])

In [40]:
def transpose_output(X: torch.Tensor, num_heads: int) -> torch.Tensor:
    """Reverse the operation of transpose_qkv."""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)

attention_output = transpose_output(attention, num_heads)
attention_output

tensor([[[0.2078, 0.1400, 0.5057, 0.1862],
         [0.2167, 0.1450, 0.4960, 0.1849]],

        [[0.8614, 0.1796, 0.8018, 0.8053],
         [0.8654, 0.1748, 0.8042, 0.8077]]])